# LogisChain AI — 02: Feature Engineering

Build the full 50+ feature set across supply chain, financial, and fusion domains.

**Goals:**
- Extract all supply chain features (network, shipment, demand, disruption)
- Extract financial features (CCC, credit risk, trade finance)
- Compute cross-domain fusion features (the LogisChain composite score)
- Validate and save to FeatureStore

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.data.feature_store import FeatureStore

gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
print('Data generated.')

In [ ]:
# Run full feature pipeline
pipeline = FeaturePipeline()
fused = pipeline.run(
    carriers=data['carriers'],
    shipments=data['shipments'],
    financial=data['financial'],
)
print(f'Fusion features shape: {fused.shape}')
print('\nFeature groups:')
sc_cols = [c for c in fused.columns if any(k in c for k in ['delay', 'on_time', 'centrality', 'congestion'])]
fin_cols = [c for c in fused.columns if any(k in c for k in ['ccc', 'altman', 'credit', 'dso'])]
fusion_cols = [c for c in fused.columns if any(k in c for k in ['logischain', 'sc_risk', 'logistics_disruption'])]
print(f'SC features ({len(sc_cols)}): {sc_cols[:5]}...')
print(f'Financial features ({len(fin_cols)}): {fin_cols[:5]}...')
print(f'Fusion features ({len(fusion_cols)}): {fusion_cols}')

In [ ]:
# Save to feature store
store = FeatureStore(store_path='../data/features')
store.save(fused, name='logischain_full_features', version='v1', metadata={'n_features': fused.shape[1]})
print(store.list_features())

In [ ]:
# Inspect LogisChain composite risk score
import matplotlib.pyplot as plt
if 'logischain_composite_risk_score' in fused.columns:
    plt.figure(figsize=(8, 4))
    plt.hist(fused['logischain_composite_risk_score'].dropna(), bins=40, color='crimson', alpha=0.7)
    plt.xlabel('LogisChain Composite Risk Score')
    plt.title('Distribution of the Flagship Fusion Feature')
    plt.show()
    print(fused['logischain_composite_risk_score'].describe())